# Test notebook lazy pipeline

In [ ]:
# Installed packages
import numpy as np
import pandas as pd

# Testing
from IPython.utils.capture import capture_output

# Our package
from data_profiling import ProfileReport

In [ ]:
# A local stand-in shaped like the Titanic dataset
# (the tests no longer download data).
rng = np.random.default_rng(1912)
n = 300
age = np.round(rng.normal(30, 14, n).clip(0.4, 80), 1)
age[rng.random(n) < 0.2] = np.nan
df = pd.DataFrame(
    {
        "PassengerId": np.arange(1, n + 1),
        "Survived": rng.integers(0, 2, n),
        "Pclass": rng.choice([1, 2, 3], n, p=[0.24, 0.21, 0.55]),
        "Name": [f"Passenger, Mr. Person {i}" for i in range(n)],
        "Sex": rng.choice(["male", "female"], n, p=[0.65, 0.35]),
        "Age": age,
        "SibSp": rng.choice([0, 1, 2, 3], n, p=[0.68, 0.23, 0.06, 0.03]),
        "Parch": rng.choice([0, 1, 2], n, p=[0.76, 0.13, 0.11]),
        "Fare": np.round(rng.lognormal(2.8, 1.0, n), 4),
        "Embarked": rng.choice(["S", "C", "Q"], n, p=[0.72, 0.19, 0.09]),
    }
)

In [ ]:
# Generate the Profiling Report (with progress bar)
with capture_output() as out:
    profile = ProfileReport(df, title="Titanic Dataset", progress_bar=True, lazy=False)

assert all(
    any(v in s.data["text/plain"] for v in ["%|", "FloatProgress"]) for s in out.outputs
)
assert len(out.outputs) == 2

In [ ]:
# Generate the Profiling Report (without progress bar)
with capture_output() as out:
    profile = df.profile_report(
        title="Titanic Dataset",
        html={"style": {"full_width": True}},
        progress_bar=True,
        lazy=True,
    )

assert len(out.outputs) == 0

with capture_output() as out:
    _ = profile.to_html()


assert all(
    any(v in s.data["text/plain"] for v in ["%|", "FloatProgress"]) for s in out.outputs
)
assert len(out.outputs) == 3

with capture_output() as out:
    _ = profile.to_file("/tmp/tmpfile.html")

assert "Export report to file" in out.outputs[0].data["text/plain"]
assert len(out.outputs) == 1

In [ ]:
# Test caching of the iterative building process
with capture_output() as out:
    profile = ProfileReport(df, title="Titanic Dataset", progress_bar=True, lazy=True)
assert len(out.outputs) == 0

with capture_output() as out:
    profile.description_set
assert len(out.outputs) == 1

with capture_output() as out:
    profile.report
assert len(out.outputs) == 1

with capture_output() as out:
    profile.html
assert len(out.outputs) == 1

with capture_output() as out:
    profile.config.html.style.theme = "united"
    profile.invalidate_cache("rendering")
    profile.to_file("/tmp/cache1.html")
assert len(out.outputs) == 2

with capture_output() as out:
    profile.config.pool_size = 1
    profile.html
assert len(out.outputs) == 0

with capture_output() as out:
    profile.config.pool_size = 0
    profile.config.samples.head = 5
    profile.config.samples.tail = 15
    profile.invalidate_cache()
    profile.to_file("/tmp/cache2.html")
assert len(out.outputs) == 4